# GPU Showcase — torch/jax/cupy vs CPU

Interactive demos comparing CPU and GPU compute with this repo's stack
(torch **cu130**, jax **cuda13**, cupy **cuda13x**) and `ipywidgets`.

**How to run** (VS Code devcontainer): start `jupyter lab` — VS Code
auto-forwards the port — open this notebook and run all cells. The widgets
are live only with a running kernel; moving a slider re-runs its callback.

On a CPU-only session (e.g. CI) every GPU path degrades gracefully and is
labeled **CPU-only session — run locally for GPU numbers**.

In [ ]:
import time, torch, numpy as np
import jax, jax.numpy as jnp

HAS_TORCH_GPU = torch.cuda.is_available()
try:
    import cupy as cp
    HAS_CUPY_GPU = bool(cp.cuda.runtime.getDeviceCount())
except Exception:
    cp, HAS_CUPY_GPU = None, False
HAS_JAX_GPU = any(d.platform != "cpu" for d in jax.devices())
GPU_NAME = torch.cuda.get_device_name(0) if HAS_TORCH_GPU else "CPU-only session"
print(f"torch GPU: {HAS_TORCH_GPU} | jax GPU: {HAS_JAX_GPU} | cupy GPU: {HAS_CUPY_GPU}")
print(f"device: {GPU_NAME}")

## Timing helpers — proper device sync before/after each rep

In [ ]:
def bench_torch(fn, warmup=1, repeat=3):
    gpu = torch.cuda.is_available()
    for _ in range(warmup):
        fn()
    best = float("inf")
    for _ in range(repeat):
        if gpu:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if gpu:
            torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)
    return best


def bench_jax(fn, warmup=1, repeat=3):
    for _ in range(warmup):
        jax.block_until_ready(fn())
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        jax.block_until_ready(fn())
        best = min(best, time.perf_counter() - t0)
    return best


def bench_generic(fn, sync=None, warmup=1, repeat=3):
    for _ in range(warmup):
        fn()
    if sync:
        sync()
    best = float("inf")
    for _ in range(repeat):
        if sync:
            sync()
        t0 = time.perf_counter()
        fn()
        if sync:
            sync()
        best = min(best, time.perf_counter() - t0)
    return best


print("helpers ready: bench_torch / bench_jax / bench_generic (warmup=1, repeat=3, min)")

## Baseline — single big matmul (N=4096)

In [ ]:
import matplotlib.pyplot as plt

N = 4096
a_cpu, b_cpu = torch.randn(N, N), torch.randn(N, N)
rows = [("torch CPU", bench_torch(lambda: a_cpu @ b_cpu))]
if HAS_TORCH_GPU:
    a_gpu, b_gpu = a_cpu.cuda(), b_cpu.cuda()
    rows.append(("torch GPU", bench_torch(lambda: a_gpu @ b_gpu)))
else:
    print("CPU-only session — run locally for GPU numbers")

fig, ax = plt.subplots(figsize=(7, 2.5))
names, times = [r[0] for r in rows], [r[1] for r in rows]
ax.barh(names, times, color=["#7f8fbf", "#e2725b"][: len(rows)])
ax.set_xlabel("seconds (min of 3)")
title = f"matmul {N}x{N} fp32 — ~{2 * N**3 / 1e9:.0f} GFLOP per call"
if len(rows) == 2:
    title += f" — speedup {times[0] / times[1]:.1f}x"
ax.set_title(title)
for bar, t in zip(ax.patches, times):
    ax.text(bar.get_width() * 1.02, bar.get_y() + bar.get_height() / 2,
            f"{t:.3f}s", va="center")
plt.show()

## Widget 1 — live matmul dueler (torch CPU vs GPU)

Drag the slider; each move re-runs the duel on a fresh pair of matrices.

In [ ]:
import ipywidgets as widgets


def duel(n=4096):
    n = int(n)
    a, b = torch.randn(n, n), torch.randn(n, n)
    rows = [("torch CPU", bench_torch(lambda: a @ b))]
    if HAS_TORCH_GPU:
        ag, bg = a.cuda(), b.cuda()
        rows.append(("torch GPU", bench_torch(lambda: ag @ bg)))
    fig, ax = plt.subplots(figsize=(6.5, 2.5))
    names, times = [r[0] for r in rows], [r[1] for r in rows]
    ax.barh(names, times, color=["#7f8fbf", "#e2725b"][: len(rows)])
    label = "  |  ".join(
        f"{nm}: {t * 1e3:.1f} ms, {2 * n**3 / t / 1e9:.0f} GFLOP/s" for nm, t in rows
    )
    ax.set_title(f"matmul {n}x{n}\n{label}", fontsize=10)
    if not HAS_TORCH_GPU:
        ax.set_xlabel("CPU-only session — run locally for GPU numbers")
    plt.show()


widgets.interact(duel, n=widgets.IntSlider(min=512, max=8192, step=512, value=4096));
duel(4096)

## Widget 2 — backend selector (numpy / torch / jax / cupy)

The dropdown filters to backends that actually work in this session.

In [ ]:
BACKENDS_AVAILABLE = ["numpy", "torch", "jax"] + (["cupy"] if HAS_CUPY_GPU else [])


def run_backend(backend="numpy", n=2048):
    n = int(n)
    if backend == "numpy":
        a, b = np.random.rand(n, n), np.random.rand(n, n)
        t = bench_generic(lambda: a @ b)
    elif backend == "torch":
        dev = "cuda" if HAS_TORCH_GPU else "cpu"
        a, b = torch.randn(n, n, device=dev), torch.randn(n, n, device=dev)
        t = bench_torch(lambda: a @ b)
    elif backend == "jax":
        key = jax.random.PRNGKey(0)
        a, b = jax.random.normal(key, (n, n)), jax.random.normal(key, (n, n))
        t = bench_jax(lambda: a @ b)
    elif backend == "cupy":
        a, b = cp.random.rand(n, n), cp.random.rand(n, n)
        t = bench_generic(lambda: a @ b, sync=cp.cuda.runtime.deviceSynchronize)
    print(f"{backend:6s} matmul {n}x{n}: {t * 1e3:9.1f} ms — {2 * n**3 / t / 1e9:5.0f} GFLOP/s")


widgets.interact(
    run_backend,
    backend=widgets.Dropdown(options=BACKENDS_AVAILABLE, value="numpy"),
    n=widgets.IntSlider(min=1024, max=6144, step=1024, value=2048),
);
run_backend(BACKENDS_AVAILABLE[0], 2048)

## Widget 3 — Monte Carlo π (vectorized, numpy CPU vs torch)

Bandwidth-bound workload: watch the GPU stretch its lead as sample count grows.

In [ ]:
def monte_pi(log10_samples=7):
    n = int(10 ** int(log10_samples))
    chunk = min(n, 2_000_000)
    nchunks = max(1, n // chunk)
    pi = 3.141592653589793

    t0 = time.perf_counter()
    counts_np = []
    for _ in range(nchunks):
        x, y = np.random.random(chunk), np.random.random(chunk)
        counts_np.append(int(((x * x + y * y) <= 1.0).sum()))
    t_np = time.perf_counter() - t0

    dev = "cuda" if HAS_TORCH_GPU else "cpu"
    torch.manual_seed(0)
    t0 = time.perf_counter()
    counts_t = []
    for _ in range(nchunks):
        x, y = torch.rand(chunk, device=dev), torch.rand(chunk, device=dev)
        counts_t.append(int(((x * x + y * y) <= 1.0).sum()))
    if dev == "cuda":
        torch.cuda.synchronize()
    t_t = time.perf_counter() - t0

    est_np = sum(counts_np) / (nchunks * chunk) * 4
    est_t = sum(counts_t) / (nchunks * chunk) * 4
    tlabel = "torch GPU" if HAS_TORCH_GPU else "torch CPU (CPU-only session — run locally for GPU numbers)"
    print(f"samples: {n:,} in {nchunks} chunk(s)")
    print(f"numpy  : pi≈{est_np:.5f}  err={abs(est_np - pi):.2e}  {t_np:.3f}s")
    print(f"{tlabel}: pi≈{est_t:.5f}  err={abs(est_t - pi):.2e}  {t_t:.3f}s")
    if HAS_TORCH_GPU:
        print(f"speedup: {t_np / t_t:.1f}x")

    xs = np.arange(1, nchunks + 1)
    run_np = 4 * np.cumsum(counts_np) / (xs * chunk)
    run_t = 4 * np.cumsum(counts_t) / (xs * chunk)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(xs, run_np, "o-", label="numpy CPU")
    ax.plot(xs, run_t, "s-", label="torch " + ("GPU" if HAS_TORCH_GPU else "CPU"))
    ax.axhline(pi, color="k", ls="--", lw=1, label="π")
    ax.set_xscale("log")
    ax.set_xlabel("chunk")
    ax.set_ylabel("running π estimate")
    ax.legend(fontsize=8)
    plt.show()


widgets.interact(monte_pi, log10_samples=widgets.IntSlider(min=5, max=8, value=7));
monte_pi(7)

## Widget 4 — mini training throughput (linear-regression step)

One autograd step (forward MSE + backward + SGD) on tensors of the chosen batch size.

In [ ]:
def train_bench(batch=16384):
    batch = int(batch)
    dim = 64

    def steps_per_sec(dev):
        X = torch.randn(batch, dim, device=dev)
        w_true = torch.randn(dim, 1, device=dev)
        y = X @ w_true + 0.1
        w = torch.zeros(dim, 1, device=dev, requires_grad=True)
        b = torch.zeros(1, device=dev, requires_grad=True)
        opt = torch.optim.SGD([w, b], lr=1e-3)

        def step():
            opt.zero_grad()
            loss = ((X @ w + b - y) ** 2).mean()
            loss.backward()
            opt.step()

        t = bench_torch(step)
        return batch / t, t

    sps_cpu, t_cpu = steps_per_sec("cpu")
    rows = [(f"torch CPU — {sps_cpu:,.0f} steps/s", t_cpu)]
    if HAS_TORCH_GPU:
        sps_gpu, t_gpu = steps_per_sec("cuda")
        rows.append((f"torch GPU — {sps_gpu:,.0f} steps/s", t_gpu))
    fig, ax = plt.subplots(figsize=(6.5, 2.2))
    names, times = [r[0] for r in rows], [r[1] for r in rows]
    ax.barh(names, times, color=["#7f8fbf", "#e2725b"][: len(rows)])
    ax.set_xlabel(f"seconds per step (batch={batch:,})")
    if not HAS_TORCH_GPU:
        ax.set_title("CPU-only session — run locally for GPU numbers", fontsize=10)
    plt.show()


widgets.interact(train_bench, batch=widgets.IntSlider(min=1024, max=65536, step=4096, value=16384));
train_bench(16384)

## Scaling sweep — matmul time vs N (log-log, matplotlib + altair)

In [ ]:
import pandas as pd
import altair as alt

NS = [256, 512, 1024, 2048, 4096, 8192]
cpu_t, gpu_t = [], []
for n in NS:
    a, b = torch.randn(n, n), torch.randn(n, n)
    cpu_t.append(bench_torch(lambda: a @ b))
    if HAS_TORCH_GPU:
        ag, bg = a.cuda(), b.cuda()
        gpu_t.append(bench_torch(lambda: ag @ bg))

df = pd.DataFrame({"n": NS, "backend": "torch CPU", "seconds": cpu_t})
enc = lambda: {
    "x": alt.X("n:Q", scale=alt.Scale(type="log")),
    "y": alt.Y("seconds:Q", scale=alt.Scale(type="log")),
}
charts = [alt.Chart(df).mark_line(point=True).encode(**enc(), color=alt.value("#7f8fbf"))]
if HAS_TORCH_GPU:
    df_gpu = pd.DataFrame({"n": NS, "backend": "torch GPU", "seconds": gpu_t})
    df = pd.concat([df, df_gpu], ignore_index=True)
    charts.append(
        alt.Chart(df_gpu).mark_line(point=True).encode(**enc(), color=alt.value("#e2725b"))
    )

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(NS, cpu_t, "o-", label="torch CPU")
if HAS_TORCH_GPU:
    ax.loglog(NS, gpu_t, "s-", label="torch GPU")
ax.set_xlabel("N")
ax.set_ylabel("seconds (min of 3)")
ax.set_title(f"matmul scaling — {GPU_NAME}")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()

alt.layer(*charts).interactive()

## Summary — expected speedups (RTX 2060, FP32)

| workload | compute bound | typical speedup |
|---|---|---|
| big matmul (N ≥ 4096) | ~6.5 TFLOPS GPU vs a few hundred GFLOPS CPU | **10-30x** |
| Monte Carlo π (bandwidth bound) | ~300 GB/s GPU vs ~30 GB/s CPU | **5-15x** |
| training step (batch ≥ 16k) | matmul-dominated | **5-20x** |

Every GPU path above is guarded: in the CI's CPU-only runner the cells fall
back to CPU and print **CPU-only session — run locally for GPU numbers**.
In your local devcontainer (NVIDIA GPU + WSL driver bind-mount) the same
notebook lights up torch-CUDA / jax-cuda13 / cupy-cuda13x automatically.